In [5]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
%%capture

!pip install transformers
!pip install datasets
!pip install evaluate

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import os

In [8]:
dataset = load_dataset("json", data_files={"train": "/content/train.jsonl", "test": "/content/validation.jsonl"})

In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 453
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 114
    })
})

In [10]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

mapDict = {
    "No Hate Speech": 0,
    "Hate Speech": 1
}

def transform_labels(label):
  label = label['completion']
  result = []
  for l in label:
    result.append(mapDict[l])

  return {"label": result}

def tokenize_function(example):
  return tokenizer(example['prompt'], padding=True, truncation=True)

In [11]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.map(transform_labels, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments

output_dir = "./bert-hate-speech-test"

training_args = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    weight_decay = 0.01,
    logging_dir = "./logs",
    logging_steps = 100,
    eval_strategy = "steps",
    eval_steps = 200,
    save_total_limit = 2,
    save_steps = 200,
    load_best_model_at_end = True,
    metric_for_best_model = "accuracy",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
from transformers import AutoModelForSequenceClassification, Trainer

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 3)

os.environ['WANDB_DISABLE'] = "true"
os.environ['WANDB_MODE'] = "offline"


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metric(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis = -1)
  return metric.compute(predictions = predictions, references = labels)

In [15]:
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['test'],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metric
)

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=171, training_loss=0.6271985371907552, metrics={'train_runtime': 1153.383, 'train_samples_per_second': 1.178, 'train_steps_per_second': 0.148, 'total_flos': 47489916326328.0, 'train_loss': 0.6271985371907552, 'epoch': 3.0})

In [17]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6598536968231201,
 'eval_accuracy': 0.6228070175438597,
 'eval_runtime': 27.5247,
 'eval_samples_per_second': 4.142,
 'eval_steps_per_second': 0.545,
 'epoch': 3.0}

In [18]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
trainer.push_to_hub("guilchaves/bert-hatespeech")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ch-test/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...ch-test/training_args.bin:  47%|####6     | 2.43kB / 5.20kB            

CommitInfo(commit_url='https://huggingface.co/guilchaves/bert-hate-speech-test/commit/050c0c86a815edb350c2d311963f344c2d3f7471', commit_message='guilchaves/bert-hatespeech', commit_description='', oid='050c0c86a815edb350c2d311963f344c2d3f7471', pr_url=None, repo_url=RepoUrl('https://huggingface.co/guilchaves/bert-hate-speech-test', endpoint='https://huggingface.co', repo_type='model', repo_id='guilchaves/bert-hate-speech-test'), pr_revision=None, pr_num=None)

In [40]:
from transformers import pipeline

pipe = pipeline("text-classification", model="guilchaves/bert-hate-speech-test")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]